<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Proceso de Poisson
### T2.1 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Motivación: conteo de eventos aleatorios
2. Definición axiomática del Proceso de Poisson
3. Distribución del número de eventos N(t)
4. Tiempos entre llegadas: distribución Exponencial
5. Tiempo hasta el n-ésimo evento: distribución Erlang
6. Superposición y descomposición
7. Verificación empírica con datos reales

## Motivación
<div class='defn'>
Un <strong>proceso estocástico de conteo</strong> {N(t), t ≥ 0} registra el número acumulado de eventos ocurridos hasta el tiempo t.
</div>

**Ejemplos en ingeniería industrial:**
- Llegadas de clientes a un banco (por hora)
- Fallas de una máquina (por semana)
- Solicitudes a un servidor web (por segundo)
- Pedidos a un centro de distribución (por día)

**Pregunta clave:** ¿Bajo qué condiciones el proceso de conteo tiene propiedades matemáticas manejables?

## Definición axiomática
El proceso {N(t), t ≥ 0} es un **Proceso de Poisson** con tasa λ > 0 si:

1. **N(0) = 0** — inicia sin eventos
2. **Incrementos independientes** — eventos en intervalos disjuntos son independientes
3. **Incrementos estacionarios** — la distribución de N(t+s) − N(t) solo depende de s
4. **Regularidad:** P(un evento en (t, t+h]) = λh + o(h)
5. **No simultaneidad:** P(dos o más eventos en (t, t+h]) = o(h)

<div class='nota'>
Las condiciones 4 y 5 implican que en un intervalo muy pequeño ocurre a lo sumo un evento con probabilidad proporcional a λh.
</div>

## Distribución de N(t)
<div class='defn'>
Si {N(t)} es un PP(λ), entonces para cualquier t > 0:

$$N(t) \sim \text{Poisson}(\lambda t)$$

$$P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}, \quad k = 0, 1, 2, \ldots$$
</div>

**Media:** E[N(t)] = λt
**Varianza:** Var[N(t)] = λt
**Coeficiente de variación:** CV = 1/√(λt) → 0 cuando t → ∞

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

lam = 3   # eventos por hora
ts = [1, 2, 5]
ks = np.arange(0, 20)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, t in zip(axes, ts):
    pmf = poisson.pmf(ks, lam*t)
    ax.bar(ks, pmf, color='#1A7A9A', alpha=0.85, edgecolor='white')
    ax.axvline(lam*t, color='#C0392B', lw=2, ls='--', label=f'Media={lam*t}')
    ax.set_title(f't = {t} h  →  N(t) ~ Poisson({lam*t})')
    ax.set_xlabel('k'); ax.set_ylabel('P(N(t)=k)')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Tiempos entre llegadas
<div class='defn'>
Si {N(t)} es PP(λ), los tiempos entre eventos consecutivos T₁, T₂, T₃, ... son **i.i.d. Exponencial(λ)**.
</div>

$$f_{T_i}(t) = \lambda e^{-\lambda t}, \quad t \geq 0$$

**Propiedad sin memoria (clave):**
$$P(T > t + s \mid T > t) = P(T > s)$$

El tiempo restante hasta el próximo evento no depende de cuánto tiempo ha pasado. Esta propiedad es **exclusiva** de la distribución Exponencial entre las distribuciones continuas.

## Tiempo hasta el n-ésimo evento
<div class='defn'>
El tiempo hasta el n-ésimo evento $S_n = T_1 + T_2 + \cdots + T_n$ sigue una distribución **Erlang-n**:

$$S_n \sim \text{Erlang}(n, \lambda) = \text{Gamma}(n, 1/\lambda)$$

$$f_{S_n}(t) = \frac{\lambda^n t^{n-1} e^{-\lambda t}}{(n-1)!}, \quad t \geq 0$$
</div>

**Media:** E[Sₙ] = n/λ
**Varianza:** Var[Sₙ] = n/λ²

## Superposición y descomposición
**Superposición (fusión):**
Si PP₁(λ₁) y PP₂(λ₂) son independientes, su unión es PP(λ₁ + λ₂).

**Descomposición (thinning):**
Si cada evento de PP(λ) se clasifica como tipo A con prob. p o tipo B con prob. 1−p, entonces:
- Eventos tipo A ∼ PP(pλ)
- Eventos tipo B ∼ PP((1−p)λ)
- Los dos procesos resultantes son **independientes**

<div class='nota'>
Estas propiedades son fundamentales para modelar redes de colas, sistemas de manufactura con múltiples productos y redes de comunicaciones.
</div>

In [ ]:
# Verificación empírica: generar realizaciones y comparar
np.random.seed(7)
lam_real = 5    # tasa verdadera (desconocida en la práctica)
n_eventos = 500

# Simular tiempos entre llegadas
interarrivals = np.random.exponential(1/lam_real, size=n_eventos)
llegadas = np.cumsum(interarrivals)

# Contar en ventanas de 1 hora
T_max = llegadas[-1]
ventanas = np.arange(0, T_max, 1.0)
conteos = [np.sum((llegadas >= v) & (llegadas < v+1)) for v in ventanas]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Comparar distribución empírica vs Poisson(lambda)
lam_est = np.mean(conteos)
k_vals = np.arange(0, max(conteos)+1)
axes[0].hist(conteos, bins=range(max(conteos)+2), density=True,
             color='#1A7A9A', alpha=0.7, edgecolor='white', label='Empírico')
axes[0].plot(k_vals, poisson.pmf(k_vals, lam_est), 'o-',
             color='#C0392B', lw=2, label=f'Poisson({lam_est:.2f})')
axes[0].set_xlabel('N(1 hora)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Conteos por ventana de 1 hora'); axes[0].legend()

# Q-Q plot de interarrivals vs Exponencial
from scipy import stats
probs = (np.arange(1, n_eventos+1) - 0.5) / n_eventos
teor = stats.expon.ppf(probs, scale=1/lam_est)
axes[1].scatter(np.sort(interarrivals), teor, alpha=0.3, color='#1A7A9A', s=8)
lim = max(np.sort(interarrivals)[-10], teor[-10])
axes[1].plot([0, lim], [0, lim], 'r--', lw=2, label='y = x')
axes[1].set_xlabel('Cuantiles empíricos'); axes[1].set_ylabel('Cuantiles Exponencial')
axes[1].set_title('Q-Q Plot: tiempos entre llegadas'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"lambda estimada = {lam_est:.3f}  (verdadera = {lam_real})")

## Conclusiones
- El Proceso de Poisson modela conteos de eventos raros, independientes y estacionarios.
- La distribución Poisson(λt) describe el número de eventos en [0, t].
- Los tiempos entre eventos son Exponencial(λ) — con la crucial **propiedad sin memoria**.
- La superposición y descomposición facilitan el modelado de sistemas complejos.
- En la práctica, verificar con histogramas, Q-Q plots y pruebas de bondad de ajuste.

**Siguiente tema:** Cadenas de Markov de Tiempo Continuo (CMTC).